# Fraud Detection Baseline Notebook

이 노트북은 train.csv 기준으로 **EDA → Feature Engineering → 모델 선정 → 학습 및 반복 개선** 순서의 baseline pipeline을 담고 있습니다.

구성:
1. 환경 설정 및 데이터 로드
2. EDA
3. Feature Engineering
4. Validation 전략
5. LightGBM Baseline 학습
6. Feature Importance 확인
7. 다음 반복 개선 아이디어


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb


## 1. 데이터 로드

- 파일명은 `train.csv` 기준
- 필요하면 경로를 수정해서 사용


In [3]:
TRAIN_PATH = 'csv/train.csv'

TARGET = 'fraud_bool'
ID_COL = 'id'

train = pd.read_csv(TRAIN_PATH)
print(train.shape)
train.head()


(700000, 33)


,id,fraud_bool,yearly_income,name_email_sim,prev_addr_months,curr_addr_months,age_bucket,days_since_req,init_transfer_amt,payment_type,...,has_other_cards,req_credit_limit,is_foreign_req,application_source,session_length_min,device_os,is_session_persistent,device_email_cnt_8w,device_fraud_history,month_idx
0,0,0,0.8,0.996036,9,19,40,0.019167,-0.795951,AB,...,1,510.0,0,INTERNET,4.125364,other,1,1,0,6
1,1,0,0.8,0.520470,12,10,30,0.010994,-0.667975,AD,...,0,1500.0,0,INTERNET,8.147967,macintosh,0,1,0,4
2,2,0,0.5,0.792133,162,11,40,70.144985,-1.051204,AC,...,0,200.0,0,INTERNET,4.996239,linux,1,1,0,3
3,3,0,0.1,0.846394,24,7,50,0.005721,-1.404540,AB,...,0,200.0,0,INTERNET,5.860972,linux,1,1,0,6
4,4,0,0.9,0.232846,-1,81,40,0.023110,-0.519206,AD,...,0,200.0,0,INTERNET,12.764765,windows,0,1,0,4


In [4]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 700000 entries, 0 to 699999
Data columns (total 33 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   id                     700000 non-null  int64  
 1   fraud_bool             700000 non-null  int64  
 2   yearly_income          700000 non-null  float64
 3   name_email_sim         700000 non-null  float64
 4   prev_addr_months       700000 non-null  int64  
 5   curr_addr_months       700000 non-null  int64  
 6   age_bucket             700000 non-null  int64  
 7   days_since_req         700000 non-null  float64
 8   init_transfer_amt      700000 non-null  float64
 9   payment_type           700000 non-null  str    
 10  zip_req_count_4w       700000 non-null  int64  
 11  req_rate_6h            700000 non-null  float64
 12  req_rate_24h           700000 non-null  float64
 13  req_rate_4w            700000 non-null  float64
 14  branch_req_count_8w    700000 non-null  int64  

## 2. EDA

baseline 단계에서는 **빠르게 확인해야 하는 것** 위주로 봅니다.

- target imbalance
- numeric / categorical 구분
- 결측치 여부
- 범주형 cardinality
- 기본 통계량


In [ ]:
print('Target distribution')
display(train[TARGET].value_counts())
display(train[TARGET].value_counts(normalize=True))

In [ ]:
num_cols = train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = train.select_dtypes(include=['object']).columns.tolist()

for col in [TARGET, ID_COL]:
    if col in num_cols:
        num_cols.remove(col)

print('Numeric columns:', len(num_cols))
print(num_cols)
print('\nCategorical columns:', len(cat_cols))
print(cat_cols)


In [ ]:
missing = train.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print('Missing columns count:', len(missing))
display(missing.head(20))


In [ ]:
display(train[num_cols].describe().T)


In [ ]:
cat_summary = pd.DataFrame({
    'nunique': train[cat_cols].nunique(),
    'sample_values': [train[col].astype(str).unique()[:5].tolist() for col in cat_cols]
})
display(cat_summary)


## 3. Feature Engineering

baseline에서는 복잡한 target encoding보다 먼저 **안전하고 빠른 파생 변수**를 만듭니다.

아이디어:
- 단기/장기 요청 빈도 비율
- 주소 안정성
- 소득 대비 요청 한도
- 디바이스 리스크
- 세션 길이 대비 활동량


In [ ]:
df = train.copy()

# 1) request rate ratio features
df['req_rate_ratio_6h_24h'] = df['req_rate_6h'] / (df['req_rate_24h'] + 1e-5)
df['req_rate_ratio_24h_4w'] = df['req_rate_24h'] / (df['req_rate_4w'] + 1e-5)

# 2) address stability
df['addr_stability'] = df['curr_addr_months'] / (df['prev_addr_months'] + 1)
df['total_addr_months'] = df['curr_addr_months'] + df['prev_addr_months']

# 3) finance-related ratios
df['credit_per_income'] = df['req_credit_limit'] / (df['yearly_income'] + 1e-5)
df['transfer_per_income'] = df['init_transfer_amt'] / (df['yearly_income'] + 1e-5)

# 4) device risk
df['device_risk'] = df['device_fraud_history'] * df['device_email_cnt_8w']

# 5) session behavior
df['session_per_req24h'] = df['session_length_min'] / (df['req_rate_24h'] + 1e-5)
df['session_per_req6h'] = df['session_length_min'] / (df['req_rate_6h'] + 1e-5)

# 6) contact validity summary
df['valid_contact_count'] = df['is_home_phone_valid'] + df['is_mobile_valid']

# 7) request pressure summary
df['request_pressure'] = (
    df['zip_req_count_4w'] +
    df['branch_req_count_8w'] +
    df['dob_email_count_4w'] +
    df['device_email_cnt_8w']
)

print(df.shape)


### 3-1. 범주형 인코딩

LightGBM은 category dtype도 다룰 수 있지만, baseline에서는 재현성과 단순성을 위해 **Label Encoding**으로 시작합니다.

나중에 비교해볼 만한 확장:
- frequency encoding
- target encoding (반드시 fold-safe)
- CatBoost 사용


In [ ]:
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

df.head()


In [ ]:
FEATURES = [col for col in df.columns if col not in [TARGET, ID_COL]]
print('Number of features:', len(FEATURES))
FEATURES


## 4. Validation 전략

현재 baseline은 **StratifiedKFold**를 사용합니다.

주의:
- fraud 문제는 실제 서비스 환경에서 시간 누수(time leakage)에 민감할 수 있음
- `month_idx`가 있으므로 나중에는 **time-based validation**도 반드시 비교하는 것이 좋음


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X = df[FEATURES]
y = df[TARGET]

pos = y.sum()
neg = len(y) - pos
scale_pos_weight = neg / pos
print('scale_pos_weight:', scale_pos_weight)


## 5. LightGBM Baseline 모델

이 baseline의 장점:
- tabular 데이터에서 강함
- 빠름
- feature importance 확인 쉬움
- 반복 실험에 유리함


In [ ]:
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbosity': -1,
    'random_state': 42,
    'scale_pos_weight': scale_pos_weight,
    'n_estimators': 2000
}

params


In [ ]:
oof_preds = np.zeros(len(df))
feature_importances = []
fold_scores = []
threshold = 0.5

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    print(f'===== Fold {fold} =====')

    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = lgb.LGBMClassifier(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)]
    )

    valid_pred = model.predict_proba(X_valid)[:, 1]
    oof_preds[valid_idx] = valid_pred
    valid_pred_label = (valid_pred >= threshold).astype(int)

    fold_precision = precision_score(y_valid, valid_pred_label, zero_division=0)
    fold_scores.append(fold_precision)
    print(f'Fold {fold} Precision: {fold_precision:.6f}')

    fi = pd.DataFrame({
        'feature': FEATURES,
        'importance': model.feature_importances_,
        'fold': fold
    })
    feature_importances.append(fi)

oof_pred_labels = (oof_preds >= threshold).astype(int)
overall_precision = precision_score(y, oof_pred_labels, zero_division=0)
print('\n===== CV Result =====')
print('Fold Precisions:', [round(s, 6) for s in fold_scores])
print('Mean Precision:', np.mean(fold_scores))
print('OOF Precision :', overall_precision)


## 6. Feature Importance

여기서 중요한 것은 단순히 높은 importance를 보는 것이 아니라:
- 어떤 파생 변수가 실제로 도움이 되었는지
- 불필요한 feature가 무엇인지
- 다음 FE iteration 방향이 어디인지

를 판단하는 것입니다.


In [ ]:
fi_df = pd.concat(feature_importances, axis=0)
fi_mean = fi_df.groupby('feature')['importance'].mean().sort_values(ascending=False)
display(fi_mean.head(30).to_frame('mean_importance'))


## 7. Submission용 test 처리 템플릿

test.csv가 있을 때는 다음 순서로 처리합니다.

1. 동일한 feature engineering 적용
2. train에서 fit한 encoder로 transform
3. 각 fold 모델 평균 예측

아래 코드는 템플릿입니다. 실제 제출 시 경로와 파일명을 맞춰 사용하세요.


In [ ]:
# TEST_PATH = 'test.csv'
# test = pd.read_csv(TEST_PATH)
#
# # 동일한 FE 적용
# test['req_rate_ratio_6h_24h'] = test['req_rate_6h'] / (test['req_rate_24h'] + 1e-5)
# test['req_rate_ratio_24h_4w'] = test['req_rate_24h'] / (test['req_rate_4w'] + 1e-5)
# test['addr_stability'] = test['curr_addr_months'] / (test['prev_addr_months'] + 1)
# test['total_addr_months'] = test['curr_addr_months'] + test['prev_addr_months']
# test['credit_per_income'] = test['req_credit_limit'] / (test['yearly_income'] + 1e-5)
# test['transfer_per_income'] = test['init_transfer_amt'] / (test['yearly_income'] + 1e-5)
# test['device_risk'] = test['device_fraud_history'] * test['device_email_cnt_8w']
# test['session_per_req24h'] = test['session_length_min'] / (test['req_rate_24h'] + 1e-5)
# test['session_per_req6h'] = test['session_length_min'] / (test['req_rate_6h'] + 1e-5)
# test['valid_contact_count'] = test['is_home_phone_valid'] + test['is_mobile_valid']
# test['request_pressure'] = (
#     test['zip_req_count_4w'] +
#     test['branch_req_count_8w'] +
#     test['dob_email_count_4w'] +
#     test['device_email_cnt_8w']
# )
#
# # categorical transform
# for col in cat_cols:
#     known_classes = set(label_encoders[col].classes_)
#     test[col] = test[col].astype(str).apply(lambda x: x if x in known_classes else 'UNK')
#     if 'UNK' not in label_encoders[col].classes_:
#         label_encoders[col].classes_ = np.append(label_encoders[col].classes_, 'UNK')
#     test[col] = label_encoders[col].transform(test[col])
#
# X_test = test[FEATURES]
#
# # fold 평균 prediction 예시
# # 실제로는 각 fold model을 저장해두고 평균내기


## 8. 다음 반복에서 해볼 것

### Feature Engineering
- `device_os`, `payment_type`, `employment_status` frequency encoding
- `month_idx` 기준 시계열 split 비교
- `yearly_income` / `req_credit_limit` binning
- interaction feature: `is_foreign_req × device_fraud_history`
- group-based stats (단, leakage 주의)

### Model
- CatBoost 비교
- XGBoost 비교
- threshold tuning (대회 metric에 따라)

### Validation
- StratifiedKFold vs month-based split 성능 비교
- public LB와 local CV gap 체크

### 분석 관점
- fraud detection은 단순 분류가 아니라 **비정상 행동 패턴 탐지**에 가까움
- 그래서 raw feature보다 **비율, 변화량, 불일치, 희소 패턴** 계열 파생변수가 중요해지는 경우가 많음
